## Extraction des features nécessaires à la création des DataFrames depuis  
- les fichiers .csv contenant chaque transaction bitcoin effectué chaque jour en 2024
- le fichier .csv recensant toutes les adresses dont ont connait l'entité

In [2]:
import pandas as pd
import numpy as np
import glob
import time
import os
from collections import defaultdict

# Création des dossiers de sortie si non existants
out_dir = "data_processed/2024_all"
os.makedirs(out_dir, exist_ok=True)

print("Modules chargés et dossiers préparés.")

Modules chargés et dossiers préparés.


#### Création à partir du .csv reliant les adresses à leur entité de deux dictionnaires, un reliant les adresses à leur entité, l'autre les entités à leur label.

In [3]:
# Chargement du mapping des entités
entities_path = "data_raw/entities/active_2024_addresses.csv"
entities_df = pd.read_csv(entities_path)

# Nettoyage des chaînes de caractères
entities_df["address"] = entities_df["address"].astype(str).str.strip()
entities_df["wallet"] = entities_df["wallet"].astype(str).str.strip()
entities_df["category"] = entities_df["category"].astype(str).str.strip()

# Création des dictionnaires
address_to_wallet = dict(zip(entities_df["address"], entities_df["wallet"]))

wallet_to_label = (
    entities_df.drop_duplicates("wallet")
    .set_index("wallet")["category"]
    .to_dict()
)

print(f"Nombre total d’adresses labellisées : {len(address_to_wallet)}")
print(f"Nombre total d’entités connues : {len(wallet_to_label)}")

Nombre total d’adresses labellisées : 408601
Nombre total d’entités connues : 73


#### Définition de la fonction nous permettant d'extraire les données de la chaîne de caractères correspondant aux inputs/outputs d'une transaction.

In [13]:
# Convertit la chaîne brute en une liste de tuples (adresse, montant)
def parse_io_field(s):
    if pd.isna(s) or not str(s).strip():
        return []

    result = []
    for part in str(s).split(";"):
        part = part.strip()
        if "," not in part:
            continue

        # Séparation de l'adresse et du montant
        addr, amt = part.split(",", 1)
        addr = addr.strip()

        # Filtrage des pseudo-adresses
        if addr.startswith("d-"):
            continue

        # Conversion du montant en entier
        try:
            amount = int(amt.strip())
        except ValueError:
            continue

        result.append((addr, amount))

    return result

#### Initialisation des dictionnaires à mettre à jour à chaque transaction.

In [ ]:
from collections import defaultdict

# Initialisation des accumulateurs globaux pour df_entity
ent_received = defaultdict(int)
ent_sent = defaultdict(int)
ent_tx_recv = defaultdict(set)
ent_tx_sent = defaultdict(set)
ent_recv_from = defaultdict(set)
ent_sent_to = defaultdict(set)

# Initialisation des accumulateurs pour df_address
addr_stats = defaultdict(lambda: {'amount_recv': 0, 'amount_sent': 0, 'tx_count': 0, 'siblings_set': set()})

# Initialisation des listes pour df_motif1 et les ponts de df_motif2
list_motif1 = []
list_outputs_ponts = []
list_inputs_ponts = []

# Dictionnaire pour le calcul de la récurrence des motifs (n_similar_txs)
relation_counter = defaultdict(int)

#### Traitement de chaques transactions.

In [ ]:
start_time = time.time()

# Recherche de tous les fichiers de l'année 2024 dans les sous-dossiers
tx_files = sorted(glob.glob("data_raw/transactions/2024/*/blockchair_bitcoin_2024*.csv"))
print(f"Nombre total de fichiers à traiter : {len(tx_files)}")

for path in tx_files:
    print(f"Traitement en cours : {os.path.basename(path)}")
    
    # Lecture par chunks pour préserver la RAM
    for chunk in pd.read_csv(path, usecols=["transaction_hash", "block_id", "inputs", "outputs"], chunksize=200000):
        
        for txh, block_id, ins, outs in zip(chunk["transaction_hash"], chunk["block_id"], chunk["inputs"], chunk["outputs"]):
            
            in_list = parse_io_field(ins)
            out_list = parse_io_field(outs)
            
            # Calcul global de la transaction (fee)
            total_in = sum(amt for _, amt in in_list)
            total_out = sum(amt for _, amt in out_list)
            tx_fee = max(0, total_in - total_out)
            
            # Structures temporaires pour grouper par entité au sein de cette transaction
            dict_in = defaultdict(lambda: {'amount': 0, 'addrs': set()})
            dict_out = defaultdict(lambda: {'amount': 0, 'addrs': set()})
            
            all_in_addrs_in_tx = {a for a, _ in in_list}
            
            # Traitement des INPUTS
            for addr, amt in in_list:
                wallet = address_to_wallet.get(addr)
                if wallet:
                    # Mise à jour des stats de l'entité
                    ent_sent[wallet] += amt
                    ent_tx_sent[wallet].add(txh)
                    dict_in[wallet]['amount'] += amt
                    dict_in[wallet]['addrs'].add(addr)
                    
                    # Mise à jour des stats de l'adresse
                    addr_stats[addr]['amount_sent'] += amt
                    addr_stats[addr]['tx_count'] += 1
                    # Les siblings sont toutes les autres adresses en input de cette même tx
                    addr_stats[addr]['siblings_set'].update(all_in_addrs_in_tx - {addr})

            # Traitement des OUTPUTS
            for addr, amt in out_list:
                wallet = address_to_wallet.get(addr)
                if wallet:
                    # Mise à jour des stats de l'entité
                    ent_received[wallet] += amt
                    ent_tx_recv[wallet].add(txh)
                    dict_out[wallet]['amount'] += amt
                    dict_out[wallet]['addrs'].add(addr)
                    
                    # Mise à jour des stats de l'adresse
                    addr_stats[addr]['amount_recv'] += amt
                    addr_stats[addr]['tx_count'] += 1

            # Création des 1-Motifs (Produit cartésien Entités IN x Entités OUT)
            for w_in, data_in in dict_in.items():
                for w_out, data_out in dict_out.items():
                    
                    # Mise à jour du graphe de relation des entités
                    if w_in != w_out:
                        ent_sent_to[w_in].update(data_out['addrs'])
                        ent_recv_from[w_out].update(data_in['addrs'])
                    
                    is_direct_loop = 1 if w_in == w_out else 0
                    relation_counter[(w_in, w_out)] += 1
                    
                    # Identifiant unique du motif pour faire la jointure plus tard
                    motif_id = f"{txh}_{w_in}_{w_out}"
                    
                    list_motif1.append({
                        'motif_id': motif_id,
                        'entity_id_sent': w_in,
                        'entity_id_recv': w_out,
                        'amount_sent': data_in['amount'],
                        'amount_received': data_out['amount'],
                        'fee': tx_fee,
                        'n_distinct_addr_sent': len(data_in['addrs']),
                        'n_distinct_addr_received': len(data_out['addrs']),
                        'is_direct_loop': is_direct_loop,
                        'tx_hash': txh
                    })
                    
                    # Sauvegarde des ponts pour construire le 2-Motif ultérieurement
                    for a_out in data_out['addrs']:
                        list_outputs_ponts.append({'motif_id': motif_id, 'adresse_charniere': a_out, 'block_id': block_id})
                    
                    for a_in in data_in['addrs']:
                        list_inputs_ponts.append({'motif_id': motif_id, 'adresse_charniere': a_in, 'block_id': block_id})

print(f"Extraction terminée en {(time.time() - start_time)/60:.2f} minutes.")

Nombre total de fichiers à traiter : 30
Traitement en cours : blockchair_bitcoin_20240401.csv
Traitement en cours : blockchair_bitcoin_20240402.csv
Traitement en cours : blockchair_bitcoin_20240403.csv
Traitement en cours : blockchair_bitcoin_20240404.csv
Traitement en cours : blockchair_bitcoin_20240405.csv
Traitement en cours : blockchair_bitcoin_20240406.csv
Traitement en cours : blockchair_bitcoin_20240407.csv
Traitement en cours : blockchair_bitcoin_20240408.csv
Traitement en cours : blockchair_bitcoin_20240409.csv
Traitement en cours : blockchair_bitcoin_20240410.csv
Traitement en cours : blockchair_bitcoin_20240411.csv
Traitement en cours : blockchair_bitcoin_20240412.csv
Traitement en cours : blockchair_bitcoin_20240413.csv
Traitement en cours : blockchair_bitcoin_20240414.csv
Traitement en cours : blockchair_bitcoin_20240415.csv
Traitement en cours : blockchair_bitcoin_20240416.csv
Traitement en cours : blockchair_bitcoin_20240417.csv
Traitement en cours : blockchair_bitcoin_2

#### Création des DataFrames à partir des dictionnaires.

In [16]:
out_dir = "data_processed/2024_all"
os.makedirs(out_dir, exist_ok=True)

# Création du df_entity
wallets = list(wallet_to_label.keys())
df_entity = pd.DataFrame({
    "entity_id": wallets,
    "label": [wallet_to_label[w] for w in wallets],
    "amount_received": [ent_received[w] for w in wallets],
    "amount_sent": [ent_sent[w] for w in wallets],
    "balance": [ent_received[w] - ent_sent[w] for w in wallets],
    "n_tx_received": [len(ent_tx_recv[w]) for w in wallets],
    "n_tx_sent": [len(ent_tx_sent[w]) for w in wallets],
    "n_addr_received_from": [len(ent_recv_from[w]) for w in wallets],
    "n_addr_sent_to": [len(ent_sent_to[w]) for w in wallets],
})
df_entity.to_csv(f"{out_dir}/df_entity.csv", index=False)
print("df_entity sauvegardé.")

# Création du df_address
addr_data = []
for addr, stats in addr_stats.items():
    addr_data.append({
        'address_id': addr,
        'entity_id': address_to_wallet[addr],
        'addr_amount_received': stats['amount_recv'],
        'addr_amount_sent': stats['amount_sent'],
        'addr_balance': stats['amount_recv'] - stats['amount_sent'],
        'tx_count': stats['tx_count'],
        # uniqueness: 1 si l'adresse n'a été vue que dans 1 seule tx, 0 sinon
        'uniqueness': 1 if stats['tx_count'] == 1 else 0,
        # siblings: le nombre d'autres adresses vues en co-input
        'siblings': len(stats['siblings_set'])
    })
df_address = pd.DataFrame(addr_data)
df_address.to_csv(f"{out_dir}/df_address.csv", index=False)
print("df_address sauvegardé.")

# Création du df_motif1
df_motif1 = pd.DataFrame(list_motif1)
if not df_motif1.empty:
    # Ajout de la feature n_similar_txs calculée grâce au relation_counter
    df_motif1['n_similar_txs'] = df_motif1.apply(lambda row: relation_counter[(row['entity_id_sent'], row['entity_id_recv'])], axis=1)
    
    # On renomme entity_id_sent en entity_id pour simplifier les futures jointures ML
    df_motif1.rename(columns={'entity_id_sent': 'entity_id'}, inplace=True)
    df_motif1.to_csv(f"{out_dir}/df_motif1.csv", index=False)
    print("df_motif1 sauvegardé.")

# Sauvegarde des Ponts (pour le df_motif2)
df_out_ponts = pd.DataFrame(list_outputs_ponts)
df_in_ponts = pd.DataFrame(list_inputs_ponts)

df_out_ponts.to_csv(f"{out_dir}/outputs_ponts.csv", index=False)
df_in_ponts.to_csv(f"{out_dir}/inputs_ponts.csv", index=False)
print("Fichiers ponts sauvegardés.")

print("Toutes les données ont été extraites avec succès !")

df_entity sauvegardé.
df_address sauvegardé.
df_motif1 sauvegardé.
Fichiers ponts sauvegardés.
Toutes les données ont été extraites avec succès !


#### On relie les 1_motifs ayant une adresse en inputs et en outputs en commun.

In [ ]:
# Chargement des données générées à l'étape précédente
in_dir = "data_processed/2024_all"

df_out_ponts = pd.read_csv(f"{in_dir}/outputs_ponts.csv")
df_in_ponts = pd.read_csv(f"{in_dir}/inputs_ponts.csv")
df_motif1 = pd.read_csv(f"{in_dir}/df_motif1.csv")

print(f"Lignes d'arrivées (Outputs) : {len(df_out_ponts)}")
print(f"Lignes de départs (Inputs) : {len(df_in_ponts)}")
print(f"Lignes Motif1 : {len(df_motif1)}")

# Fusion sur l'adresse charnière
df_liens = pd.merge(
    df_out_ponts,
    df_in_ponts,
    on='adresse_charniere',
    suffixes=('_1', '_2')
)

# Filtre chronologique : TX1 (création de l'UTXO) doit précéder TX2 (dépense)
df_liens = df_liens[df_liens['block_id_1'] <= df_liens['block_id_2']]

# Extraction des hashs de transactions depuis l'identifiant du motif (txhash_win_wout)
df_liens['tx_hash_1'] = df_liens['motif_id_1'].str.split('_').str[0]
df_liens['tx_hash_2'] = df_liens['motif_id_2'].str.split('_').str[0]

# Filtre anti-boucle : on s'assure qu'il s'agit bien de deux transactions distinctes
df_liens = df_liens[df_liens['tx_hash_1'] != df_liens['tx_hash_2']]

# Nettoyage : on ne conserve que les paires de motifs uniques
df_liens = df_liens[['motif_id_1', 'motif_id_2']].drop_duplicates()

print(f"Nombre de paires de 2-motifs valides trouvées : {len(df_liens)}")

Lignes d'arrivées (Outputs) : 1469
Lignes de départs (Inputs) : 14814
Lignes Motif1 : 1408
Nombre de paires de 2-motifs valides trouvées : 960


#### On récupère les features de df_motif1.csv et on calcule les dernières features avant de créer le DataFrame.

In [18]:
# Rapatriement des features de la première transaction (branche 1)
df_motif2 = pd.merge(df_liens, df_motif1, left_on='motif_id_1', right_on='motif_id')
df_motif2 = df_motif2.drop(columns=['motif_id'])

# Renommage des colonnes pour la branche 1
rename_dict_1 = {
    'entity_id': 'entity_id_1',
    'entity_id_recv': 'entity_id_2',
    'amount_sent': 'amount_sent_1',
    'amount_received': 'amount_received_1',
    'fee': 'fee_1',
    'n_distinct_addr_sent': 'n_distinct_addr_sent_1',
    'n_distinct_addr_received': 'n_distinct_addr_received_1',
    'is_direct_loop': 'is_direct_loop_1',
    'n_similar_txs': 'n_similar_txs_1',
    'tx_hash': 'tx_hash_1' # Ajout pour corriger le bug
}
df_motif2 = df_motif2.rename(columns=rename_dict_1)

# Rapatriement des features de la deuxième transaction (branche 2)
df_motif2 = pd.merge(df_motif2, df_motif1, left_on='motif_id_2', right_on='motif_id')
df_motif2 = df_motif2.drop(columns=['motif_id'])

# Renommage des colonnes pour la branche 2
rename_dict_2 = {
    'entity_id': 'entity_id_2_check',
    'entity_id_recv': 'entity_id_3',
    'amount_sent': 'amount_sent_2',
    'amount_received': 'amount_received_2',
    'fee': 'fee_2',
    'n_distinct_addr_sent': 'n_distinct_addr_sent_2',
    'n_distinct_addr_received': 'n_distinct_addr_received_2',
    'is_direct_loop': 'is_direct_loop_2',
    'n_similar_txs': 'n_similar_txs_2',
    'tx_hash': 'tx_hash_2' # Ajout pour corriger le bug
}
df_motif2 = df_motif2.rename(columns=rename_dict_2)

# Calcul des features globales du 2-motif
# direct_loop_whole : 1 si l'entité de départ (1) est la même que l'entité d'arrivée (3)
df_motif2['direct_loop_whole'] = (df_motif2['entity_id_1'] == df_motif2['entity_id_3']).astype(int)

# direct_distinct_whole : l'inverse de la boucle globale
df_motif2['direct_distinct_whole'] = 1 - df_motif2['direct_loop_whole']

# Nettoyage des colonnes intermédiaires inutiles (dont les fameux tx_hash)
df_motif2 = df_motif2.drop(columns=['entity_id_2_check', 'tx_hash_1', 'tx_hash_2'])

# Pour l'apprentissage ML, l'entité principale du motif est l'entité d'origine
df_motif2 = df_motif2.rename(columns={'entity_id_1': 'entity_id'})

# Sauvegarde du fichier final
df_motif2.to_csv(f"{in_dir}/df_motif2.csv", index=False)
print(f"df_motif2 sauvegardé avec succès ({len(df_motif2)} motifs trouvés).")

df_motif2 sauvegardé avec succès (960 motifs trouvés).
